In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.dim_customer AS
SELECT xxhash64(cust_id, eff_start) AS customer_sk, cust_id, name, email, city,
       eff_start, eff_end, is_current
FROM workspace.silver.clean_customers;

CREATE OR REPLACE TABLE workspace.gold.dim_date AS
SELECT CAST(date_format(d,'yyyyMMdd') AS INT) AS date_key, d AS full_date,
       year(d) yr, month(d) mth, dayofweek(d) dow
FROM (SELECT explode(sequence(DATE'2025-10-01', DATE'2026-12-31', INTERVAL 1 DAY)) AS d);

CREATE OR REPLACE TABLE workspace.gold.fact_sales AS
SELECT s.sale_id, CAST(date_format(s.sale_ts,'yyyyMMdd') AS INT) AS date_key,
       s.store_id, s.product, s.qty, s.price, s.qty * s.price AS sales_amt
FROM workspace.silver.clean_pos_sales s;

CREATE OR REPLACE TABLE workspace.gold.agg_daily_sales AS
SELECT date_key, product, sum(sales_amt) AS total_sales, count(*) AS txn_cnt
FROM workspace.gold.fact_sales GROUP BY date_key, product;

num_affected_rows,num_inserted_rows
